# Lecture 7 — Class Exercise
## Heatmap & Waterfall: Netflix Catalogue

> **Push to:** `week07/lecture07_exercise.ipynb`

**Rules:**
1. Heatmap: colour scale must match the data type (sequential for counts, diverging for above/below)
2. Waterfall: use green for additions, red for subtractions, blue for totals
3. Insight title tells the setup-conflict-resolution story (or at minimum states the finding)
4. Annotate at least one cell or bar directly

---


In [3]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# upload the csv file first in Colab/Jupyter
df = pd.read_csv('netflix_catalogue.csv')

print(f"Loaded: {len(df)} titles")
print(df['type'].value_counts())
print(df.head())

Loaded: 3000 titles
type
Movie      1974
TV Show    1026
Name: count, dtype: int64
      type  release_year  added_year             genre        country rating  \
0    Movie          2014        2016  Sci-Fi & Fantasy         France  PG-13   
1    Movie          2010        2014     Documentaries  United States  TV-MA   
2  TV Show          2011        2012     Kids & Family  United States  TV-14   
3    Movie          2016        2018             Anime          India     PG   
4    Movie          2014        2016     Kids & Family         Canada  TV-MA   

   duration  
0       157  
1       127  
2         6  
3       134  
4        77  


## Task 1 — Heatmap: content by rating and release decade

**What to build:** A heatmap showing the number of titles by **content rating** (y-axis) and **decade** (x-axis).

**Requirements:**
- Create a 'decade' column: `df['decade'] = (df['release_year'] // 10 * 10).astype(str) + 's'`
- Filter to TV-14, TV-MA, PG-13, R, PG only (most common ratings)
- Sequential colour scale (Blues)
- Values shown in cells (`text_auto=True`)
- Insight title about which rating dominates which decade


In [4]:
# Task 1 — Heatmap: content by rating and release decade

df['decade'] = (df['release_year'] // 10 * 10).astype(str) + 's'

ratings = ['TV-14', 'TV-MA', 'PG-13', 'R', 'PG']

heatmap_data = (
    df[df['rating'].isin(ratings)]
    .groupby(['rating', 'decade'])
    .size()
    .reset_index(name='count')
)

pivot = heatmap_data.pivot(
    index='rating',
    columns='decade',
    values='count'
).fillna(0)

fig = px.imshow(
    pivot,
    text_auto=True,
    color_continuous_scale='Blues',
    labels={'color': 'Number of Titles'},
    title='TV-MA and TV-14 dominate recent Netflix decades, showing a shift toward mature content',
    width=1000,
    height=600
)

fig.update_traces(
    hovertemplate='<b>Rating:</b> %{y}<br><b>Decade:</b> %{x}<br><b>Titles:</b> %{z}<extra></extra>'
)

fig.update_layout(
    xaxis_title='Release Decade',
    yaxis_title='Content Rating',
    font=dict(family='Arial', size=12),
    margin=dict(l=80, r=40, t=80, b=60)
)

# Annotation on one important cell
fig.add_annotation(
    x='2010s',
    y='TV-MA',
    text='High mature-content volume',
    showarrow=True,
    arrowhead=2,
    ax=40,
    ay=-40
)

fig.show()


## Task 2 — Waterfall: Movie vs TV Show additions by year

**What to build:** A waterfall chart showing how Netflix's **Movie library** grew year by year (2015-2022).

**Requirements:**
- Filter to Movies only
- Group by `added_year`, count titles per year
- Final bar should be the cumulative total
- Green bars (additions), blue total
- Annotation on the year with the largest single addition
- Insight title naming the growth story


In [5]:
# Task 2 — Waterfall: Movie additions by year

movies = df[
    (df['type'] == 'Movie') &
    (df['added_year'] >= 2015) &
    (df['added_year'] <= 2022)
].copy()

adds = (
    movies.groupby('added_year')
    .size()
    .reset_index(name='new_titles')
)

cumulative_total = adds['new_titles'].sum()

largest_year = adds.loc[adds['new_titles'].idxmax(), 'added_year']
largest_value = adds['new_titles'].max()

x_vals = adds['added_year'].astype(str).tolist() + ['Total 2015-2022']
y_vals = adds['new_titles'].tolist() + [cumulative_total]
measure = ['relative'] * len(adds) + ['total']

fig = go.Figure(go.Waterfall(
    x=x_vals,
    y=y_vals,
    measure=measure,
    text=[f'{v:,}' for v in y_vals],
    textposition='outside',
    connector=dict(line=dict(color='#AAAAAA', dash='dot')),
    increasing=dict(marker_color='green'),
    decreasing=dict(marker_color='red'),
    totals=dict(marker_color='blue')
))

fig.update_layout(
    title='Netflix movie library grew steadily from 2015 to 2022, with the biggest jump in one peak year',
    xaxis_title='Year',
    yaxis_title='Movies Added',
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=12),
    showlegend=False,
    height=650
)

fig.add_annotation(
    x=str(largest_year),
    y=largest_value,
    text=f'Largest addition: {largest_value} movies',
    showarrow=True,
    arrowhead=2,
    ax=40,
    ay=-50
)

fig.show()
